# Step Counter - Model Evaluation on Test Set

Evaluate a trained model on the held-out test set.

## Steps:
1. Load test data
2. Load trained model
3. Evaluate on test set
4. Visualize results

## 1. Configuration and Imports

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Import model architectures
from src.models.shallow_cnn import ShallowCNN
from src.models.deep_cnn import DeepCNN

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Configuration

Specify which trained model to evaluate

In [ ]:
# Model to evaluate - UPDATE THIS PATH
MODEL_PATH = Path('../models/saved/deep_cnn_20240315_143022/final_model.pth')

# Data path
DATA_DIR = Path('../data/processed')

# Verify paths exist
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}')
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Data directory not found: {DATA_DIR}')

print(f'Evaluating model: {MODEL_PATH.parent.name}')

## 3. Load Trained Model

In [ ]:
# Load checkpoint
checkpoint = torch.load(MODEL_PATH, map_location=device)

# Extract config and info
config = checkpoint['config']
input_shape = checkpoint['input_shape']
task_type = checkpoint.get('task_type', 'regression')

print(f'Model Configuration:')
print(f'  Task: {task_type}')
print(f'  Input shape: {input_shape}')
print(f'  Model type: {config.get("model_type", "unknown")}')

# Determine model architecture
model_type = str(config.get('experiment_name', '')).split('_')[0]

if model_type == 'shallow' or 'shallow' in str(config.get('model_type', '')):
    model = ShallowCNN(
        input_channels=input_shape[1],
        sequence_length=input_shape[0],
        num_filters=config.get('n_filters', 64),
        dropout_rate=config.get('dropout_rate', 0.5)
    )
elif model_type == 'deep' or 'deep' in str(config.get('model_type', '')):
    model = DeepCNN(
        input_channels=input_shape[1],
        sequence_length=input_shape[0],
        num_filters=config.get('n_filters', 64),
        dropout_rate=config.get('dropout_rate', 0.5)
    )
else:
    raise ValueError(f'Unknown model type: {model_type}')

# Load weights
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

print(f'\nModel loaded: {model_type.upper()}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 4. Load Test Data

In [ ]:
# Load test data
test_data = np.load(DATA_DIR / 'cnn_test_data.npz')
X_test = test_data['X']
y_test = test_data['y_count'].astype(np.float32)

# Convert to PyTorch tensors
X_test_tensor = torch.FloatTensor(X_test).permute(0, 2, 1)  # (N, L, C) -> (N, C, L)
y_test_tensor = torch.FloatTensor(y_test)

# Create DataLoader
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f'Test Data Loaded:')
print(f'  X_test: {X_test.shape} -> Tensor: {X_test_tensor.shape}')
print(f'  y_test: {y_test.shape}')
print(f'  Task: regression (step counting)')
print(f'  Batches: {len(test_loader)}')

## 5. Evaluate on Test Set

In [ ]:
# Make predictions
test_preds = []
test_targets = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X).squeeze()
        
        test_preds.extend(outputs.cpu().numpy())
        test_targets.extend(batch_y.numpy())

test_preds = np.array(test_preds)
test_targets = np.array(test_targets)

print(f'\n{"="*60}')
print(f'TEST SET EVALUATION')
print(f'{"="*60}')

# Regression metrics
test_mae = np.abs(test_preds - test_targets).mean()
test_rmse = np.sqrt(((test_preds - test_targets) ** 2).mean())
test_r2 = 1 - (np.sum((test_targets - test_preds) ** 2) / np.sum((test_targets - test_targets.mean()) ** 2))

criterion = nn.MSELoss()
test_loss = criterion(torch.FloatTensor(test_preds), torch.FloatTensor(test_targets)).item()

print(f'\nMetrics:')
print(f'  Loss (MSE): {test_loss:.4f}')
print(f'  MAE: {test_mae:.4f}')
print(f'  RMSE: {test_rmse:.4f}')
print(f'  R² Score: {test_r2:.4f}')

# Additional info
test_pred_rounded = np.maximum(np.round(test_preds).astype(int), 0)
exact_match = (test_targets == test_pred_rounded).mean()
total_error = test_pred_rounded.sum() - test_targets.sum()

print(f'\nStep Count Accuracy:')
print(f'  Exact Match: {exact_match:.4f} ({exact_match*100:.1f}%)')
print(f'  Total True Steps: {int(test_targets.sum())}')
print(f'  Total Predicted Steps: {int(test_pred_rounded.sum())}')
print(f'  Total Error: {int(total_error)} steps')

print(f'\n{"="*60}')

## 6. Visualize Results

In [ ]:
# Regression visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

test_pred_rounded = np.maximum(np.round(test_preds).astype(int), 0)

# Predicted vs Actual
axes[0].scatter(test_targets, test_pred_rounded, alpha=0.3, s=10)
max_val = max(test_targets.max(), test_pred_rounded.max())
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect')
axes[0].set_xlabel('True Step Count')
axes[0].set_ylabel('Predicted Step Count')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Error Distribution
errors = test_pred_rounded - test_targets
axes[1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='r', linestyle='--', linewidth=2, label='Zero Error')
axes[1].set_xlabel('Prediction Error (Predicted - True)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Error Distribution (MAE={np.abs(errors).mean():.2f})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Error by True Count
unique_counts = np.unique(test_targets)
mae_by_count = [np.abs(errors[test_targets == c]).mean() for c in unique_counts]

axes[2].bar(unique_counts, mae_by_count, edgecolor='black', alpha=0.7)
axes[2].set_xlabel('True Step Count')
axes[2].set_ylabel('Mean Absolute Error')
axes[2].set_title('Error by Step Count')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. Save Test Results

In [ ]:
# Save test results to model directory
test_results = {
    'task_type': 'regression',
    'test_loss': float(test_loss),
    'test_mae': float(test_mae),
    'test_rmse': float(test_rmse),
    'test_r2': float(test_r2),
    'exact_match': float(exact_match),
    'total_true_steps': int(test_targets.sum()),
    'total_pred_steps': int(test_pred_rounded.sum()),
    'total_error': int(total_error)
}

# Save to model directory
test_results_path = MODEL_PATH.parent / 'test_results.json'
with open(test_results_path, 'w') as f:
    json.dump(test_results, f, indent=2)

print(f'Test results saved to: {test_results_path}')

## Summary

This notebook evaluated a trained model on the test set.

**Files created:**
- `test_results.json` - Test set metrics